# Orthomosaic — Part 2: Colour Correction

**Apply per-channel percentile stretch to the raw orthomosaic to remove sensor-induced colour cast.**

The `correct_colors` function (defined in `config_nb`) clips each RGB band to its
[2nd, 98th] percentile range of valid (non-black) pixels and rescales to uint8.
This removes the characteristic "hot-red" cast common in GoPro aerial imagery.

> **Prerequisite.** `01_sfm_orthomosaic` must have written `orthomosaic.tif` to `output_dir`.

> **Runtime.** Runs on **Serverless environment 5**.

---

**Last Update:** September 21, 2026

## Setup

In [ ]:
%run ./config_nb

## Step 1: Per-channel percentile stretch

In [ ]:
import time as _t_cc

_t0 = _t_cc.perf_counter()
try:
    correct_colors(output_tiff, corrected_tiff)
    print(f"Colour correction done in {_t_cc.perf_counter()-_t0:.1f}s → {corrected_tiff}")
except Exception as e:
    print(f"[ERROR] Colour correction failed after {_t_cc.perf_counter()-_t0:.1f}s: {e}")
    raise

## Step 2: Before / After Comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling

_MAX_PX = 800

def _load_thumb(p):
    with rasterio.open(p) as ds:
        scale = min(1.0, _MAX_PX / max(ds.width, ds.height))
        ow = max(1, int(ds.width * scale)); oh = max(1, int(ds.height * scale))
        return np.stack([ds.read(b, out_shape=(oh, ow), resampling=Resampling.average)
                         for b in range(1, 4)], axis=-1)

before = _load_thumb(output_tiff)
after  = _load_thumb(corrected_tiff)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(before); axes[0].set_title("Before: orthomosaic.tif"); axes[0].axis("off")
axes[1].imshow(after);  axes[1].set_title("After:  orthomosaic_corrected.tif"); axes[1].axis("off")
plt.tight_layout(); plt.show()

## Step 3: Interactive Map

In [ ]:
show_raster(corrected_tiff)

## Steps performed

1. **Percentile stretch** — each RGB band clipped to its 2nd–98th percentile across
   non-black pixels and rescaled to 0–255 uint8.
2. **Before / after comparison** — matplotlib side-by-side render.
3. **Interactive map** — Folium overlay on OpenStreetMap.

**Next:** Part 3 converts the corrected GeoTIFF to Cloud-Optimised GeoTIFF (COG) layout
using **GeoBrix `rst_cog_convert`**.